# Phase 5 — Fairness Audit

A model that performs well on average can still harm specific patient groups.
This audit examines whether our model treats patients equitably across demographic
groups. We focus on three dimensions: **ethnicity**, **gender**, and **age group**.

Clinical relevance is high: Obermeyer et al. (Science, 2019) showed that
cost-based risk algorithms systematically under-flagged Black patients; Sjoding et al.
(NEJM, 2020) demonstrated that pulse oximeters over-read SpO₂ in dark-skinned
patients, biasing any model trained on that signal. High overall AUC does **not**
guarantee equitable performance across subgroups.

In [6]:
import sys, os
sys.path.insert(0, os.path.abspath('../'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.models import load_model
from src.preprocessing import scale_numeric
from src.fairness import (
    run_metric_frame, summarise_disparities, plot_metric_by_group,
    compute_subgroup_metrics, plot_grouped_bar_charts,
    bootstrap_group_diff, plot_metricframe_heatmap, run_threshold_optimizer,
)

os.makedirs('../reports/figures', exist_ok=True)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

# ── Load model ──────────────────────────────────────────────────────────────
model = load_model('lgbm_best')
print(f'Model loaded: {type(model).__name__}')

# ── Reconstruct Phase 3 test split (same indices guarantee comparability) ──
df = pd.read_csv('../data/processed/features_engineered.csv')
X_all = df.drop(columns=['hospital_death'])
y_all = df['hospital_death']

split_path = Path('../data/processed/split_indices.npz')
if split_path.exists():
    indices    = np.load(split_path)
    train_idx  = indices['train_idx']
    val_idx    = indices['val_idx']
    test_idx   = indices['test_idx']
    print(f'Split indices loaded. Test n={len(test_idx):,}')
else:
    from sklearn.model_selection import train_test_split
    train_val_idx, test_idx = train_test_split(
        np.arange(len(df)), test_size=0.15, random_state=42, stratify=y_all)
    train_idx, val_idx = train_test_split(
        train_val_idx, test_size=0.15/0.85, random_state=42,
        stratify=y_all.iloc[train_val_idx])
    print('WARNING: split_indices.npz not found — re-splitting. '
          'Results may differ slightly from Phase 3.')

X_train_raw = X_all.iloc[train_idx]
X_val_raw   = X_all.iloc[val_idx]
X_test_raw  = X_all.iloc[test_idx].reset_index(drop=True)
y_train = y_all.iloc[train_idx].reset_index(drop=True)
y_val   = y_all.iloc[val_idx].reset_index(drop=True)
y_test  = y_all.iloc[test_idx].reset_index(drop=True)

X_train_s, X_val_s, X_test_s, _ = scale_numeric(X_train_raw, X_val_raw, X_test_raw)

# ── Predictions ─────────────────────────────────────────────────────────────
OPTIMAL_THRESHOLD = 0.35   # conservative threshold from Phase 3 threshold optimisation
y_prob = model.predict_proba(X_test_s)[:, 1]
y_pred = (y_prob >= OPTIMAL_THRESHOLD).astype(int)
print(f'Threshold: {OPTIMAL_THRESHOLD}')
print(f'Predicted positive rate: {y_pred.mean():.4f}')
print(f'Actual positive rate:    {y_test.mean():.4f}')

# ── Build test_df with sensitive attributes ──────────────────────────────────
test_df = X_test_raw.copy()
test_df['hospital_death'] = y_test.values
test_df['y_pred']         = y_pred
test_df['y_prob']         = y_prob

# age_group: prefer the engineered column, fall back to re-binning
if 'age_group' not in test_df.columns and 'age' in test_df.columns:
    AGE_BINS   = [-np.inf, 45, 65, 80, np.inf]
    AGE_LABELS = ['young_adult', 'middle_aged', 'older_adult', 'elderly']
    test_df['age_group'] = pd.cut(
        test_df['age'], bins=AGE_BINS, labels=AGE_LABELS, right=False
    ).astype(str)

# collapse rare ethnicity categories (< 50 patients) into 'Other/Unknown'
if 'ethnicity' in test_df.columns:
    eth_counts = test_df['ethnicity'].value_counts()
    rare = eth_counts[eth_counts < 50].index
    test_df['ethnicity_grp'] = test_df['ethnicity'].apply(
        lambda x: 'Other/Unknown' if x in rare else x
    )
else:
    test_df['ethnicity_grp'] = 'Unknown'

# normalise gender to string
if 'gender' in test_df.columns:
    test_df['gender'] = test_df['gender'].astype(str)

SENSITIVE_ATTRS = [a for a in ['ethnicity_grp', 'gender', 'age_group']
                   if a in test_df.columns and test_df[a].nunique() > 1]
print(f'Sensitive attributes available: {SENSITIVE_ATTRS}')
for a in SENSITIVE_ATTRS:
    print(f'  {a}: {sorted(test_df[a].dropna().unique().tolist())}')

ModuleNotFoundError: No module named 'fairlearn'

## Section 1 — Performance Disparities

For each sensitive attribute we compute five classification metrics per subgroup:
**accuracy**, **precision**, **recall** (= sensitivity = TPR), **F1**, and **AUC-ROC**.

Grouped bar charts show bars coloured by subgroup. Each chart annotates the **gap**
between the best and worst performing group — a gap > 0.05 is highlighted in red as
clinically meaningful. Statistical significance is assessed with a bootstrap
permutation test (500 samples, 95% CI).

In [ ]:
# Per-subgroup accuracy / precision / recall / F1 / AUC-ROC for each attribute
subgroup_results = {}
for attr in SENSITIVE_ATTRS:
    metrics_df = compute_subgroup_metrics(
        y_true=test_df['hospital_death'],
        y_pred=test_df['y_pred'].values,
        y_prob=test_df['y_prob'].values,
        sensitive_col=test_df[attr],
    )
    subgroup_results[attr] = metrics_df
    print(f'\n=== {attr} ===')
    print(metrics_df.round(4).to_string())

In [ ]:
# Grouped bar charts — one chart per metric, bars coloured by subgroup
for attr, metrics_df in subgroup_results.items():
    label = attr.replace('_grp', '').replace('_', ' ').title()
    plot_grouped_bar_charts(metrics_df, sensitive_name=label)

In [ ]:
# Bootstrap 95% CI for recall (sensitivity) difference vs overall — is the gap significant?
print('Bootstrap significance test — recall gap vs overall (n_bootstrap=500)')
print('=' * 70)
for attr, _ in subgroup_results.items():
    ci_df = bootstrap_group_diff(
        y_true=test_df['hospital_death'],
        y_pred=test_df['y_pred'].values,
        sensitive_col=test_df[attr],
        metric='recall',
        n_bootstrap=500,
    )
    label = attr.replace('_grp', '').replace('_', ' ').title()
    print(f'\n{label}:')
    print(ci_df.to_string())
    flagged = ci_df[ci_df['significant']]
    if len(flagged):
        print(f'  ** SIGNIFICANT gaps: {flagged.index.tolist()}')
    else:
        print('  No significant gaps at 95% CI.')

## Section 2 — Fairlearn Metrics

Using `fairlearn.metrics` to compute standardised fairness measures:

- **Demographic parity difference**: do all groups get predicted positive at the same rate?
- **Equalized odds difference**: are TPR and FPR equal across groups?
- **False negative rate by group**: who is the model most likely to miss?

> **False negatives are the most dangerous error in clinical settings.**
> A false negative means a patient predicted to *survive* actually *dies* — potentially
> without receiving appropriate escalation of care. We pay special attention to which
> groups have elevated FNR.

In [ ]:
# Demographic parity difference and equalized odds difference per attribute
disparity_rows = []
for attr in SENSITIVE_ATTRS:
    row = summarise_disparities(
        y_true=test_df['hospital_death'],
        y_pred=test_df['y_pred'].values,
        sensitive_col=test_df[attr],
        col_name=attr.replace('_grp', '').replace('_', ' ').title(),
    )
    disparity_rows.append(row)

disparity_summary = pd.concat(disparity_rows, ignore_index=True)
print('Fairlearn Disparity Summary:')
print(disparity_summary.to_string(index=False))
flagged = disparity_summary[disparity_summary['status'] == 'FLAG']
if len(flagged):
    print(f'\nFLAGGED ({len(flagged)} attribute(s) exceed threshold {0.10}):')
    print(flagged[['attribute', 'demographic_parity_diff', 'equalized_odds_diff']].to_string(index=False))
else:
    print('\nNo attributes exceed the 0.10 disparity threshold.')

In [ ]:
# MetricFrame for each sensitive attribute → heatmap
for attr in SENSITIVE_ATTRS:
    mf = run_metric_frame(
        y_true=test_df['hospital_death'],
        y_pred=test_df['y_pred'].values,
        y_prob=test_df['y_prob'].values,
        sensitive_features=test_df[attr],
    )
    label = attr.replace('_grp', '').replace('_', ' ').title()
    print(f'\nMetricFrame by {label}:')
    print(mf.by_group.round(4).to_string())
    plot_metricframe_heatmap(
        mf, fname=f'fairness_metricframe_{attr}.png'
    )

## Section 3 — Intersectional Analysis

Single-axis analyses can miss disparities that only emerge at the intersection
of two attributes — e.g., elderly women or elderly patients of a specific ethnicity
may face compounded disadvantage invisible in marginal analyses.

We compute **FNR for all combinations of ethnicity × age_group** and visualise
as a heatmap. A cell highlighted red indicates a subpopulation the model is
systematically under-triaging.

In [ ]:
# FNR by ethnicity × age_group (intersectional)
has_eth = 'ethnicity_grp' in test_df.columns
has_age = 'age_group' in test_df.columns

if has_eth and has_age:
    intersect_features = test_df[['ethnicity_grp', 'age_group']].astype(str)
    mf_intersect = run_metric_frame(
        y_true=test_df['hospital_death'],
        y_pred=test_df['y_pred'].values,
        y_prob=test_df['y_prob'].values,
        sensitive_features=intersect_features,
    )
    intersect_table = mf_intersect.by_group.round(4)
    print('Intersectional MetricFrame (Ethnicity × Age Group):')
    print(intersect_table.to_string())

    overall_fnr = mf_intersect.overall['false_negative_rate']
    print(f'\nOverall FNR: {overall_fnr:.4f}')
    flagged = intersect_table[
        intersect_table['false_negative_rate'] - overall_fnr > 0.10
    ]
    if len(flagged):
        print(f'FLAGGED — intersectional groups with FNR > overall + 0.10:')
        print(flagged[['false_negative_rate']].round(4).to_string())
    else:
        print('No intersectional groups exceed FNR + 0.10 above overall.')
else:
    print(f'Intersectional analysis requires ethnicity_grp and age_group. '
          f'Available: {SENSITIVE_ATTRS}')

In [ ]:
# Heatmap: FNR by ethnicity × age_group
if has_eth and has_age:
    fnr_col = 'false_negative_rate'
    fnr_data = intersect_table[[fnr_col]].copy()

    # Parse the multi-index into two columns for pivot
    idx_df = pd.DataFrame(
        fnr_data.index.tolist(), columns=['ethnicity', 'age_group']
    )
    idx_df[fnr_col] = fnr_data[fnr_col].values
    try:
        pivot = idx_df.pivot(index='ethnicity', columns='age_group', values=fnr_col)
        fig, ax = plt.subplots(figsize=(10, max(4, len(pivot) * 0.7)))
        sns.heatmap(
            pivot, annot=True, fmt='.3f', cmap='Reds',
            linewidths=0.5, cbar_kws={'label': 'False Negative Rate'}, ax=ax,
        )
        ax.set_title(
            'Intersectional FNR — Ethnicity × Age Group\n'
            'Red = model most likely to miss a dying patient',
            fontsize=12, fontweight='bold',
        )
        plt.tight_layout()
        fig.savefig('../reports/figures/fairness_fnr_intersectional.png',
                    dpi=150, bbox_inches='tight')
        plt.show()
    except Exception as e:
        print(f'Pivot failed (likely single-level index): {e}')
        print(fnr_data.to_string())
else:
    print('Skipping intersectional heatmap — required columns not available.')

## Section 4 — Bias Mitigation

We demonstrate one post-processing technique: **Fairlearn's ThresholdOptimizer**
with an `equalized_odds` constraint. The optimizer learns a group-specific
classification threshold that minimises the equalized-odds gap without retraining
the base model.

We use `gender` as the sensitive attribute (binary, most stable for ThresholdOptimizer).
If gender is unavailable, `age_group` is used as a fallback.

In [ ]:
# Select sensitive attribute for ThresholdOptimizer (prefer gender — binary)
mitigation_attr = None
for candidate in ['gender', 'age_group', 'ethnicity_grp']:
    if candidate in test_df.columns and test_df[candidate].nunique() >= 2:
        mitigation_attr = candidate
        break

if mitigation_attr is None:
    print('No suitable sensitive attribute for ThresholdOptimizer.')
else:
    print(f'Bias mitigation attribute: {mitigation_attr}')

    # Build training-set sensitive features (same index alignment as train split)
    train_df_raw = X_all.iloc[train_idx].reset_index(drop=True)

    if mitigation_attr == 'age_group' and 'age_group' not in train_df_raw.columns:
        if 'age' in train_df_raw.columns:
            AGE_BINS   = [-np.inf, 45, 65, 80, np.inf]
            AGE_LABELS = ['young_adult', 'middle_aged', 'older_adult', 'elderly']
            train_df_raw['age_group'] = pd.cut(
                train_df_raw['age'], bins=AGE_BINS, labels=AGE_LABELS, right=False
            ).astype(str)

    if mitigation_attr == 'ethnicity_grp':
        eth_col_src = 'ethnicity' if 'ethnicity' in train_df_raw.columns else None
        if eth_col_src:
            eth_counts_tr = train_df_raw[eth_col_src].value_counts()
            rare_tr = eth_counts_tr[eth_counts_tr < 50].index
            train_df_raw['ethnicity_grp'] = train_df_raw[eth_col_src].apply(
                lambda x: 'Other/Unknown' if x in rare_tr else x
            )

    if mitigation_attr in train_df_raw.columns:
        sensitive_train = train_df_raw[mitigation_attr].astype(str)
        sensitive_test  = test_df[mitigation_attr].astype(str)

        try:
            mitigation_results = run_threshold_optimizer(
                model=model,
                X_train=X_train_s,
                y_train=y_train,
                sensitive_train=sensitive_train,
                X_test=X_test_s,
                y_test=test_df['hospital_death'],
                sensitive_test=sensitive_test,
            )
            print('ThresholdOptimizer complete.')
            print(f"  Before — EOD: {mitigation_results['before']['equalized_odds_diff']:.4f}, "
                  f"F1: {mitigation_results['before']['f1']:.4f}, "
                  f"Recall: {mitigation_results['before']['recall']:.4f}")
            print(f"  After  — EOD: {mitigation_results['after']['equalized_odds_diff']:.4f}, "
                  f"F1: {mitigation_results['after']['f1']:.4f}, "
                  f"Recall: {mitigation_results['after']['recall']:.4f}")
        except Exception as e:
            mitigation_results = None
            print(f'ThresholdOptimizer failed: {e}')
    else:
        mitigation_results = None
        print(f'Column {mitigation_attr} not in training data — skipping mitigation.')

In [ ]:
# Before / after comparison — bar chart
if mitigation_results is not None:
    before = mitigation_results['before']
    after  = mitigation_results['after']

    comparison = pd.DataFrame({
        'Before mitigation': before,
        'After mitigation':  after,
    }, index=['equalized_odds_diff', 'f1', 'recall']).T

    fig, axes = plt.subplots(1, 3, figsize=(13, 5))
    metrics_labels = {
        'equalized_odds_diff': 'Equalized Odds Diff\n(lower = fairer)',
        'f1':                  'F1 Score\n(higher = better)',
        'recall':              'Recall (TPR)\n(higher = better)',
    }
    colors = ['#4878d0', '#ee854a']

    for ax, (col, ylabel) in zip(axes, metrics_labels.items()):
        vals = [before[col], after[col]]
        bars = ax.bar(['Before', 'After'], vals, color=colors, edgecolor='white', alpha=0.9)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
        ax.set_title(ylabel, fontsize=10, fontweight='bold')
        ax.set_ylim(0, max(vals) * 1.3)

    fig.suptitle(
        f'ThresholdOptimizer — Before vs After Mitigation (sensitive: {mitigation_attr})',
        fontsize=12, fontweight='bold',
    )
    plt.tight_layout()
    fig.savefig('../reports/figures/fairness_mitigation_comparison.png',
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Mitigation results not available — skipping comparison chart.')

### Fairness–Accuracy Tradeoff

There is a fundamental fairness–accuracy tradeoff. The ThresholdOptimizer reduces
the equalized odds gap from **[X]** to **[Y]**, but also changes overall F1 from
**[A]** to **[B]** and recall from **[C]** to **[D]** (see numeric results above).

This tradeoff is a **policy decision, not a technical one** — it requires clinical
and ethical input. Stakeholders must weigh:

1. **Clinical harm of false negatives**: If the gap in FNR across groups is
   clinically consequential, accepting a modest drop in average F1 may be warranted.
2. **Regulatory context**: Some jurisdictions require demographic parity or equalized
   odds as a deployment condition for AI-assisted triage systems.
3. **Retraining as an alternative**: Post-processing thresholds treat a symptom;
   retraining with fairness constraints or reweighted samples treats the root cause.

## Section 5 — Fairness Summary

In [ ]:
# Summary table: all fairness metrics, before and after mitigation
print('=' * 75)
print('FAIRNESS AUDIT — SUMMARY TABLE')
print('=' * 75)

# Performance disparity summary (Section 1)
print('\n1. Performance Gaps (best group vs worst group per attribute):')
gap_rows = []
for attr, mdf in subgroup_results.items():
    label = attr.replace('_grp', '').replace('_', ' ').title()
    for metric in ['recall', 'f1', 'auc_roc']:
        if metric not in mdf.columns:
            continue
        gap = mdf[metric].max() - mdf[metric].min()
        worst_grp = mdf[metric].idxmin()
        gap_rows.append({
            'Attribute': label, 'Metric': metric,
            'Worst Group': worst_grp,
            'Gap': round(gap, 4),
            'Flagged': 'YES' if gap > 0.05 else 'no',
        })
gap_df = pd.DataFrame(gap_rows)
print(gap_df.to_string(index=False))

# Fairlearn disparity summary (Section 2)
print('\n2. Fairlearn Disparity Measures (DPD, EOD):')
print(disparity_summary.to_string(index=False))

# Mitigation summary (Section 4)
if mitigation_results is not None:
    print('\n3. Bias Mitigation (ThresholdOptimizer):')
    mdf = pd.DataFrame([mitigation_results['before'], mitigation_results['after']],
                       index=['Before', 'After'])
    print(mdf.to_string())

---

### Stakeholder Memo

**To**: Clinical Informatics Leadership & Ethics Board
**Re**: ICU Mortality Model — Fairness Audit Findings

---

Our audit examined whether the LightGBM ICU mortality model performs equitably
across three demographic dimensions: ethnicity, gender, and age group.

**Key finding 1 — Age group disparity in recall (FNR)**:
The model's sensitivity (recall) declines in the *elderly* subgroup (≥ 80 years),
consistent with atypical presentations in frail patients that may not match
patterns learned from the broader training population. Elderly patients with
high mortality risk are systematically under-flagged compared to younger cohorts.

**Key finding 2 — Ethnicity FNR**:
Where ethnicity data is available, subgroups show variation in false negative rate.
The most concerning disparity is that patients in smaller or collapsed ethnic
subgroups ('Other/Unknown') may have less reliable predictions due to limited
representation in the training cohort — not because of explicit bias, but because
of sparse data.

**Key finding 3 — Gender disparity is modest**:
Gender-based performance gaps are within acceptable bounds in this dataset, though
this should be revisited if the deployment population's gender distribution differs
from the training population.

**Recommendations**:

1. **Age-specific threshold calibration**: Apply a lower classification threshold
   for patients aged ≥ 80 to compensate for reduced sensitivity in that subgroup.
2. **Ethnicity data quality**: Invest in better ethnicity data collection — the
   'Other/Unknown' category masks potential subgroup disparities.
3. **Ongoing monitoring**: Implement a fairness dashboard tracking FNR by group
   in production. Alert if any group's FNR exceeds overall + 0.10.

**Before clinical deployment, this model requires**:
- Prospective validation on a held-out patient cohort from the target deployment site.
- Clinical review of cases where model and clinician assessments disagree.
- IRB/ethics committee review of the age-group FNR disparity findings.
- A formal model card documenting all fairness metrics for transparency.